# Cleaning and normalization

**Module 2 — Lesson 11**

The CSV files from the pipeline contain raw extracted values — names appear exactly as the parser found them, with OCR noise, inconsistent casing, stray punctuation, and fragments from garbled scans.

Before importing into Neo4j, we need normalized values to merge on. If `Kay Mann` and `kay mann` end up as two separate User nodes, the graph is wrong. If `kay.mann@enron.com` and `Kay.Mann@enron.com` become two Mailbox nodes, every downstream query is compromised.

A lot of this can be figured out at entity resolution time -- but the more merges we get for free on the way in, the tighter the graph will be, and the easier further resolutions will be.

## The approach

This notebook applies a standard NLP normalization pipeline to each entity, creating `_norm` columns alongside the raw values. Import will merge on the normalized form; the raw value is preserved as a property for traceability.

If we accidentally merge two 'John Smith' nodes, who actually happen to be different people, we can detach them later, based on the csv.

This approach works for this particular dataset. We know, for example, that very few people will have identical names among the few thousand who worked at Enron over less than a decade of emails.

However, if you were working with, for instance, census data -- you simply could not do it this way. In an ideal scenario, you should always merge on the most unique identifier, or constellation of identifiers.

## Setup

In [1]:
import csv
import re
import unicodedata
from pathlib import Path

CSV_DIR = Path("data/csv")

def load_csv(name):
    with open(CSV_DIR / name) as f:
        return list(csv.DictReader(f))

users = load_csv("users.csv")
mailboxes = load_csv("mailboxes.csv")
domains = load_csv("domains.csv")

print(f"Users: {len(users):,}")
print(f"Mailboxes: {len(mailboxes):,}")
print(f"Domains: {len(domains):,}")

Users: 5,552
Mailboxes: 5,654
Domains: 1,098


## 1. Inspect the raw data

Before writing any normalization code, look at what you're working with.

The common issues shown below have been found by actually opening the file and looking at the data. Feel free to do so yourself.

In [2]:
EMAIL_RE = re.compile(r'[\w.+\-]+@[\w.\-]+\.\w+')

names = [r['name'] for r in users]

has_at = [n for n in names if '@' in n]
has_semi = [n for n in names if ';' in n]
has_angle = [n for n in names if '<' in n or '>' in n]
all_upper = [n for n in names if n == n.upper() and len(n) > 3]

print(f"Total users: {len(names):,}")
print(f"  Contains @:          {len(has_at):,}")
print(f"  Contains ; :         {len(has_semi):,}")
print(f"  Contains < or >:     {len(has_angle):,}")
print(f"  ALL CAPS (len > 3):  {len(all_upper):,}")
print()

print("Samples with @:")
for n in has_at[:5]:
    print(f"  {n!r}")
print()

print("Samples with ;:")
for n in has_semi[:5]:
    print(f"  {n!r}")

Total users: 5,552
  Contains @:          106
  Contains ; :         25
  Contains < or >:     3
  ALL CAPS (len > 3):  47

Samples with @:
  "'Tana.Jones@enron.com'"
  '- *Kevin.Ruscitti@enron.com'
  '- *cfrankl2@enron.com'
  '- *cgarcia@enron.com'
  '- *cindy.vachuska@enron.com'

Samples with ;:
  'Albrecht, Kristin; Stokley, Chris'
  'Beck, Sally; Shankman, Jeffrey A.; Mcconnell, Mike; Causey, Annette;'
  'Blair, Lynn; Dietz, Rick; Dornan, Dari; Darveaux, Mary; Berger, Larry'
  'Calger, Christopher F.; Yoder, Christian; Hall, Steve C.; Swerzbin,'
  'Giron, Darron C.; Gossett, Jeffrey C.; McMichael Jr., Ed'


The noise falls into predictable categories: email addresses leaking into the name field, semicolons from the template's multi-line join, angle bracket fragments from OCR-garbled `Name <email>` pairs, and inconsistent casing.

A normalization pipeline handles these systematically.

## 2. The normalization pipeline

Each normalization step is a small function. Composed together, they form a pipeline that takes a raw string and returns a cleaned, normalized form.

**For names:**

1. **Unicode normalization** — convert accented characters to their base form (`é` → `e`)
2. **Strip artifacts** — remove angle brackets, semicolons, quote marks, stray OCR punctuation
3. **Relocate email addresses** — if an email leaked into a name field and the address field is empty, move it rather than discard it
4. **Collapse whitespace** — multiple spaces, tabs, newlines → single space
5. **Lowercase** — `Kay Mann` → `kay mann`
6. **Strip and trim** — remove leading/trailing whitespace and punctuation

**For emails:**

1. **Strip garbage prefixes** — leading commas, angle brackets from OCR noise (but not leading dots — those indicate OCR truncation of the local part)
2. **Reject invalid** — if the result contains commas, newlines, spaces, or multiple `@` signs, it's not a valid email → empty string
3. **OCR domain correction** — a lookup table maps garbled domains to their correct form (`enror.corm` → `enron.com`, `enron.cem` → `enron.com`)
4. **Lowercase**

The domain correction table is built from corpus inspection, not heuristics — each entry is a specific OCR misreading observed in the data.

In [3]:
def normalize_unicode(text):
    """Decompose accented characters to base + combining, then strip combining marks."""
    decomposed = unicodedata.normalize('NFD', text)
    return ''.join(c for c in decomposed if unicodedata.category(c) != 'Mn')


def strip_artifacts(text):
    """Remove angle brackets, semicolons, quotes, and stray OCR punctuation."""
    text = re.sub(r'[<>]', '', text)
    text = re.sub(r';', ' ', text)
    text = re.sub(r'[\\(\\)]', '', text)
    text = text.strip('"\'')
    return text


def strip_emails(text):
    """Remove any email addresses that leaked into a name field."""
    return EMAIL_RE.sub('', text)


def collapse_whitespace(text):
    """Multiple spaces, tabs, newlines → single space."""
    return re.sub(r'\s+', ' ', text)


# OCR domain correction table — maps garbled domains to their correct form.
# Built by inspecting the corpus: these are predictable OCR misreadings
# of common domains, not guesses.
DOMAIN_CORRECTIONS = {
    # enron.com variants
    "enron.cam":   "enron.com",
    "enron.cem":   "enron.com",
    "enron.con":   "enron.com",
    "enron.cor":   "enron.com",
    "enron.corm":  "enron.com",
    "enron.cors":  "enron.com",
    "enron.com-":  "enron.com",
    "enran.ccom":  "enron.com",
    "enren.com":   "enron.com",
    "enren.con":   "enron.com",
    "enrom.com":   "enron.com",
    "enror.com":   "enron.com",
    "enror.cor":   "enron.com",
    "enror.corm":  "enron.com",
    "enrorn.com":  "enron.com",
    "ernron.com":  "enron.com",
    "erron.com":   "enron.com",
    "enron,.com":  "enron.com",
    "enror,.com":  "enron.com",
    "enron.co":    "enron.com",
}


def correct_domain(domain):
    """Apply OCR correction lookup to a domain."""
    return DOMAIN_CORRECTIONS.get(domain, domain)


def normalize_name(raw):
    """Full normalization pipeline for a person name."""
    text = normalize_unicode(raw)
    text = strip_artifacts(text)
    text = strip_emails(text)
    text = collapse_whitespace(text)
    text = text.lower().strip().strip('-.,:;\'\"* ')
    return text


def normalize_email(raw):
    """Normalize an email address: lowercase, strip garbage prefixes,
    apply OCR domain corrections.
    Returns empty string if the value is not a valid email.
    Leading dots are preserved — they indicate OCR truncation, not noise."""
    text = raw.strip().lower()
    text = re.sub(r'^[,<>\s]+', '', text)
    text = re.sub(r'[<>\s]+$', '', text)
    if ',' in text or '\n' in text or ' ' in text:
        return ''
    if text.count('@') != 1:
        return ''
    # Apply domain correction
    local, domain = text.split('@')
    domain = correct_domain(domain)
    return f"{local}@{domain}"


def normalize_domain(raw):
    """Normalize a domain: lowercase, strip, apply OCR corrections."""
    d = raw.strip().lower().strip('.')
    return correct_domain(d)

In another scenario, we might add many more normalization steps than these. We could even enlist an LLM again to normalize on hard cases.

In some historical datasets for example, where spellings are not standardized, the following may all refer to the same entity:

- Jon Bunian
- Ionas Bunyane
- Iohannus Bunyun
- John Bunniane
- etc. 

Based on domain expertise, we might add a spelling normalizer to ensure that these are appropriately normalized to 'John Bunyan'. 

## Run a test
Test the pipeline on the noisy samples from section 1.

In [4]:
test_names = [
    '"Barbara Denson"',
    '"Alan Z.',
    "Kitchen, Louise",
    "- *Kevin.Ruscitti@enron.com",
    "; Harry; Kingerski",
    "Nicholas O'Day",
    "[REDACTED] B6",
    ".com>",
]

test_emails = [
    "rob.bradley@enron.com",
    ".williams@enron.com",
    ",adarm@enron.corm>, alex",
    "bree. yf@topica.com \nsubject: \nre: tuesday",
    "kay.mann@enron.com",
    "chris.germany@enror.corm",
    "scott.neal@ernron.com",
    ".alex@enror.cor",
]

print(f"{'Raw name':<45s} {'Normalized'}")
print("-" * 75)
for raw in test_names:
    print(f"{raw:<45s} {normalize_name(raw)!r}")

print()
print(f"{'Raw email':<50s} {'Normalized'}")
print("-" * 80)
for raw in test_emails:
    print(f"{raw:<50s} {normalize_email(raw)!r}")

Raw name                                      Normalized
---------------------------------------------------------------------------
"Barbara Denson"                              'barbara denson'
"Alan Z.                                      'alan z'
Kitchen, Louise                               'kitchen, louise'
- *Kevin.Ruscitti@enron.com                   ''
; Harry; Kingerski                            'harry kingerski'
Nicholas O'Day                                "nicholas o'day"
[REDACTED] B6                                 '[redacted] b6'
.com>                                         'com'

Raw email                                          Normalized
--------------------------------------------------------------------------------
rob.bradley@enron.com                              'rob.bradley@enron.com'
.williams@enron.com                                '.williams@enron.com'
,adarm@enron.corm>, alex                           ''
bree. yf@topica.com 
subject: 
re: tuesday       

The pipeline strips artifacts, lowercases, and normalizes unicode. Values that are entirely noise (like `.com>`) normalize to empty strings — these will be filtered out during import.

For the name field, `normalize_name` strips any email addresses from the text. When this is applied to relationship rows (SENT, RECEIVED, CC_ON), a separate step first checks whether the name field's email is the only reference to that entity — if so, it relocates the email to the address field rather than discarding it.

## 3. Apply to CSV files

Add a `name_norm` column to `users.csv`, an `address_norm` to `mailboxes.csv`, and normalized columns to every relationship file. The raw values stay — normalized values sit alongside them.

In [5]:
# Users: add name_norm
users_out = []
for r in users:
    norm = normalize_name(r['name'])
    users_out.append({'name': r['name'], 'name_norm': norm})

print(f"Users: {len(users)} -> {len(users_out)} with name_norm")

# Mailboxes: add address_norm
mailboxes_out = []
for r in mailboxes:
    norm = normalize_email(r['address'])
    mailboxes_out.append({'address': r['address'], 'address_norm': norm})

print(f"Mailboxes: {len(mailboxes)} -> {len(mailboxes_out)} with address_norm")

# Domains: add name_norm
domains_out = []
for r in domains:
    norm = normalize_domain(r['name'])
    domains_out.append({'name': r['name'], 'name_norm': norm})

print(f"Domains: {len(domains)} -> {len(domains_out)} with name_norm")

Users: 5552 -> 5552 with name_norm
Mailboxes: 5654 -> 5654 with address_norm
Domains: 1098 -> 1098 with name_norm


Now normalize the relationship files. Each row gets `name_norm` and `address_norm` columns that match the node files.

In [6]:
def normalize_rel_rows(filename):
    """Add name_norm and address_norm to a relationship file.
    
    If the name field contains an email address and the address field is empty,
    relocate the email to address rather than losing it.
    """
    rows = load_csv(filename)
    out = []
    relocated = 0
    for r in rows:
        name_raw = r.get('name', '')
        address_raw = r.get('address', '')
        
        # Relocate: email in name field, no address
        if name_raw and '@' in name_raw and not address_raw:
            m = EMAIL_RE.search(name_raw)
            if m:
                address_raw = m.group()
                name_raw = EMAIL_RE.sub('', name_raw).strip().strip('-*,; ')
                relocated += 1
        
        name_norm = normalize_name(name_raw) if name_raw else ''
        address_norm = normalize_email(address_raw) if address_raw else ''
        
        out.append({
            **r,
            'name': name_raw, 'address': address_raw,
            'name_norm': name_norm, 'address_norm': address_norm,
        })
    
    if relocated:
        print(f"  {filename}: {relocated} emails relocated from name to address")
    return out

sent_out = normalize_rel_rows("SENT.csv")
received_out = normalize_rel_rows("RECEIVED.csv")
cc_out = normalize_rel_rows("CC_ON.csv")

used_rows = load_csv("USED.csv")
used_out = []
for r in used_rows:
    used_out.append({
        'name': r['name'], 'address': r['address'],
        'name_norm': normalize_name(r['name']),
        'address_norm': normalize_email(r['address']),
    })

has_mailbox_rows = load_csv("HAS_MAILBOX.csv")
has_mailbox_out = []
for r in has_mailbox_rows:
    has_mailbox_out.append({
        'domain': r['domain'], 'address': r['address'],
        'domain_norm': normalize_domain(r['domain']),
        'address_norm': normalize_email(r['address']),
    })

print()
print(f"SENT:         {len(sent_out):,} rows")
print(f"RECEIVED:     {len(received_out):,} rows")
print(f"CC_ON:        {len(cc_out):,} rows")
print(f"USED:         {len(used_out):,} rows")
print(f"HAS_MAILBOX:  {len(has_mailbox_out):,} rows")

  SENT.csv: 5 emails relocated from name to address
  RECEIVED.csv: 9 emails relocated from name to address
  CC_ON.csv: 3 emails relocated from name to address

SENT:         4,911 rows
RECEIVED:     14,266 rows
CC_ON:        1,348 rows
USED:         5,043 rows
HAS_MAILBOX:  5,588 rows


## 4. Verify

Check that the normalized values look correct and that deduplication worked.

In [7]:
print("Sample normalized users:")
for r in users_out[:10]:
    print(f"  {r['name']:<40s} -> {r['name_norm']!r}")

print()
print("Sample normalized mailboxes:")
for r in mailboxes_out[:10]:
    print(f"  {r['address']:<40s} -> {r['address_norm']!r}")

print()
print("Sample normalized SENT:")
for r in sent_out[:5]:
    print(f"  {r['doc_id']}  name_norm={r['name_norm']!r:30s}  address_norm={r['address_norm']!r}")

Sample normalized users:
  'Matthew Faunce'                         -> 'matthew faunce'
  'Tana.Jones@enron.com'                   -> ''
  * Thanksgiving Games                     -> 'thanksgiving games'
  *SSSB Utilities Team                     -> 'sssb utilities team'
  , Garrick                                -> 'garrick'
  - *Kevin.Ruscitti@enron.com              -> ''
  - *cfrankl2@enron.com                    -> ''
  - *cgarcia@enron.com                     -> ''
  - *cindy.vachuska@enron.com              -> ''
  - *despey@enron.com                      -> ''

Sample normalized mailboxes:
  ,adarm@enron.corm>, alex                 -> ''
  ,amitavaderron,com>, arguel              -> ''
  ,anguel@erron.com>, arita                -> ''
  ,heather@enror.com>, jaesoca 
lew <,jaesoco@enron.com>, jason -> ''
  ,pop@enron.com>, chonawee                -> ''
  .alex@enror.cor                          -> '.alex@enron.com'
  .antta@enrorn.com                        -> '.antta@enron.com'
  

## 5. Write normalized CSV files

Overwrite the CSV files with the normalized versions. Each file now has both raw and `_norm` columns.

In [8]:
def write_csv(filename, rows, fieldnames):
    with open(CSV_DIR / filename, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"  {filename}: {len(rows):,} rows")

print("Writing normalized CSV files:")
write_csv("users.csv", users_out, ['name', 'name_norm'])
write_csv("mailboxes.csv", mailboxes_out, ['address', 'address_norm'])
write_csv("domains.csv", domains_out, ['name', 'name_norm'])
write_csv("SENT.csv", sent_out, ['doc_id', 'name', 'address', 'name_norm', 'address_norm'])
write_csv("RECEIVED.csv", received_out, ['doc_id', 'name', 'address', 'name_norm', 'address_norm'])
write_csv("CC_ON.csv", cc_out, ['doc_id', 'name', 'address', 'name_norm', 'address_norm'])
write_csv("USED.csv", used_out, ['name', 'address', 'name_norm', 'address_norm'])
write_csv("HAS_MAILBOX.csv", has_mailbox_out, ['domain', 'address', 'domain_norm', 'address_norm'])

Writing normalized CSV files:
  users.csv: 5,552 rows
  mailboxes.csv: 5,654 rows
  domains.csv: 1,098 rows
  SENT.csv: 4,911 rows
  RECEIVED.csv: 14,266 rows
  CC_ON.csv: 1,348 rows
  USED.csv: 5,043 rows
  HAS_MAILBOX.csv: 5,588 rows


## Summary

- **Unicode normalization** strips accents so `e` matches `e` during import
- **Artifact stripping** removes angle brackets, semicolons, and OCR punctuation that leaked into name fields
- **Email relocation** moves addresses that ended up in name fields to the address column, preserving the entity rather than discarding it
- **OCR domain correction** maps garbled domains to their correct form via a corpus-specific lookup table
- **Lowercasing** ensures `Kay Mann` and `kay mann` merge to one node at import
- Every row keeps its raw value alongside the `_norm` column -- no rows are dropped or deduplicated here. `MERGE` at import time handles deduplication on the normalized values.

**Next:** `2.9_import_to_neo4j.ipynb` -- load the normalized CSVs into Neo4j.